In [ ]:
import os 
import polars as pl
from github import Github, Auth, UnknownObjectException, GithubException
from datetime import datetime, timezone
from settings import load_settings

In [2]:
SETTINGS = load_settings()
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")
GH = Github(auth=Auth.Token(token=GITHUB_TOKEN))

In [ ]:
init_df = pl.scan_parquet(SETTINGS.initial_dataset_path)

In [16]:
init_df.head().collect()

SnapshotAt,package_name,package_version,package_published_at,dep_name,dep_version,MinimumDepth,github_repo
datetime[μs],str,str,datetime[μs],str,str,i64,str
2023-04-10 21:01:49.219309,"""12factor-vault""","""0.1.23""",null,"""certifi""","""2022.12.7""",3,"""jdelic/12factor-vault"""
2023-04-10 21:01:49.219309,"""12factor-vault""","""0.1.23""",null,"""charset-normalizer""","""3.1.0""",3,"""jdelic/12factor-vault"""
2023-04-10 21:01:49.219309,"""12factor-vault""","""0.1.23""",null,"""django-dbconn-retry""","""0.1.7""",1,"""jdelic/12factor-vault"""
2023-04-10 21:01:49.219309,"""12factor-vault""","""0.1.23""",null,"""hvac""","""1.1.0""",1,"""jdelic/12factor-vault"""
2023-04-10 21:01:49.219309,"""12factor-vault""","""0.1.23""",null,"""idna""","""3.4.0""",3,"""jdelic/12factor-vault"""


In [17]:
unique_repos = (
    init_df
        .select(
            pl.col("package_name"),
            pl.col("github_repo"),
        )
        .unique()
        .collect()
)

In [18]:
unique_repos.shape

(42447, 2)

In [19]:
unique_repos.head()

package_name,github_repo
str,str
"""bioimageio-core""","""bioimage-io/core-bioimage-io-p…"
"""cfn-sync""","""samjarrett/cfn-sync"""
"""http-prompt""","""eliangcs/http-prompt"""
"""cobamp""","""biosystemsum/cobamp"""
"""gstools""","""geostat-framework/gstools"""


In [ ]:
def get_repo_age_and_staleness(full_name: str):
    try:
        repo = GH.get_repo(full_name)
        now = datetime.now(timezone.utc)

        created_at = repo.created_at
        pushed_at = repo.pushed_at

        repo_age_years = round((now - created_at).days / 365.2425, 2)
        commit_staleness_days = round((now - pushed_at).days, 2)

        return {"repo_age_years": repo_age_years, "commit_staleness_days": commit_staleness_days}
    except UnknownObjectException:
        return {"repo_age_years": None, "commit_staleness_days": None}
    except GithubException as e:
        # optional: handle rate limits / other GitHub API errors
        print(f"GitHub error for {full_name}: {e}")
        return {"repo_age_years": None, "commit_staleness_days": None}


In [26]:
#get_repo_age_and_staleness("jdelic/12factor-vault")

In [27]:
#get_repo_age_and_staleness_v2("jdelic/12factor-vault")

In [30]:
enriched_unique_repos = (
    unique_repos
        #.head()    
        .with_columns(pl.col("github_repo")
        .map_elements(
            lambda repo: get_repo_age_and_staleness(repo),
            return_dtype=pl.Struct(
                [
                    pl.Field("repo_age_years", pl.Float64),
                    pl.Field("commit_staleness_days", pl.Int64),
                ]
            )
        )
        .alias("repo_stats")
    )
    .with_columns(
        pl.col("repo_stats").struct.field("repo_age_years"),
        pl.col("repo_stats").struct.field("commit_staleness_days"),
    )
    .drop("repo_stats")

)

Following Github server redirection from /repos/eliangcs/http-prompt to /repositories/55584626
Following Github server redirection from /repos/ab0142/ironfence to /repositories/694985632


UnknownObjectException: 404 {"message": "Not Found", "documentation_url": "https://docs.github.com/rest/repos/repos#get-a-repository", "status": "404"}

In [29]:
enriched_unique_repos.head()

package_name,github_repo,repo_age_years,commit_staleness_days
str,str,f64,i64
"""bioimageio-core""","""bioimage-io/core-bioimage-io-p…",6.37,2
"""cfn-sync""","""samjarrett/cfn-sync""",6.06,4
"""http-prompt""","""eliangcs/http-prompt""",9.94,663
"""cobamp""","""biosystemsum/cobamp""",7.2,983
"""gstools""","""geostat-framework/gstools""",8.16,78
